In [23]:
import pandas as pd
import requests

In [39]:
def get_revisions(page_title):
    endpoint = "https://en.wikipedia.org/w/api.php"
    headers = {
        "User-Agent": "WikiResearchBot/0.1 (your_email@example.com)"
    }

    params = {
        "action": "query",
        "format": "json",
        "prop": "revisions",
        "titles": page_title,
        "rvprop": "ids|timestamp|user|comment|size",
        "rvlimit": "max",
        "formatversion": "2",
    }

    all_revisions = []

    while True:
        resp = requests.get(endpoint, params=params, headers=headers)
        data = resp.json()

        pages = data.get("query", {}).get("pages", [])
        if pages and "revisions" in pages[0]:
            all_revisions.extend(pages[0]["revisions"])

        # pagination
        if "continue" in data:
            params.update(data["continue"])
        else:
            break

    return all_revisions

def format_revisions(revisions):
    df = pd.DataFrame(revisions)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp")
    df["hour"] = df["timestamp"].dt.floor("h")

    edits = df.groupby("hour").size().rename("edits")
    unique_editors = df.groupby("hour")["user"].nunique().rename("unique_editors")

    df["seen_before"] = df["user"].duplicated()
    df["new_editor"] = (~df["seen_before"]).astype(int)
    new_editors = df.groupby("hour")["new_editor"].sum()
    features = pd.concat([df, edits, unique_editors, new_editors], axis=1).fillna(0)

    return df

In [40]:
revisions = get_revisions("Dubai chocolate")
df = format_revisions(revisions)
df.head()

,revid,parentid,user,timestamp,size,comment,temp,anon,hour,seen_before,new_editor
318,1263385884,0,Killarnee,2024-12-16 10:37:39+00:00,2751,[[WP:AES|←]]Created page with '{{Short descrip...,NaN,NaN,2024-12-16 10:00:00+00:00,False,1
317,1263387050,1263385884,Miminity,2024-12-16 10:50:37+00:00,2781,Added tags to the page using [[Wikipedia:Page ...,NaN,NaN,2024-12-16 10:00:00+00:00,False,1
316,1263390158,1263387050,Killarnee,2024-12-16 11:25:27+00:00,2751,Removed {{[[Template:Orphan|Orphan]]}} tag,NaN,NaN,2024-12-16 11:00:00+00:00,True,0
315,1263494572,1263390158,Ineedtogo,2024-12-17 00:10:38+00:00,2765,,NaN,NaN,2024-12-17 00:00:00+00:00,False,1
314,1263924518,1263494572,194.230.147.7,2024-12-19 12:13:50+00:00,2765,/* History */Typo,NaN,True,2024-12-19 12:00:00+00:00,False,1
